
# Atlas notebooks
--------------
----------
## Remote loading and building ensembles of climate projection data in climate4R.

*08/07/2021*

**M. Iturbide** (Santander Meteorology Group. Institute of Physics of Cantabria, CSIC-UC, Santander, Spain).

> This notebook describes the generation of two NetCDFs that are included as auxiliary material in this repository: `notebooks/auxiliary-material/CMIP5_historical_pr.nc` and `notebooks/auxiliary-material/CMIP5_rcp85_pr.nc`. These files contain precipitation climatologies 
of a reduced set of models for the 1986-2005 and 2041-2060 periods respectively.

> This and other notebooks are available at https://github.com/IPCC-WG1/Atlas for reproducibility and reusability purpose.  


### 1. Load packages and functions
**Packages**:

In [1]:
# Climate4R package for data loading
library(loadeR)
# Climate4R package for data visualization
# <https://doi.org/10.1016/j.envsoft.2017.09.008>
library(visualizeR)
# Dev utilities used for source_url
library(devtools)

Loading required package: rJava

Loading required package: loadeR.java

Java version 1.7x amd64 by Oracle Corporation detected

NetCDF Java Library v4.6.0-SNAPSHOT (23 Apr 2015) loaded and ready

Loading required package: climate4R.UDG

climate4R.UDG version 0.2.3 (2021-07-05) is loaded

Please use 'citation("climate4R.UDG")' to cite this package.

loadeR version 1.7.0 (2020-09-18) is loaded


Get the latest stable version (1.7.1) using <devtools::install_github(c('SantanderMetGroup/climate4R.UDG','SantanderMetGroup/loadeR'))>

Please use 'citation("loadeR")' to cite this package.

Loading required package: transformeR




    _______   ____  ___________________  __  ________ 
   / ___/ /  / /  |/  / __  /_  __/ __/ / / / / __  / 
  / /  / /  / / /|_/ / /_/ / / / / __/ / /_/ / /_/_/  
 / /__/ /__/ / /  / / __  / / / / /__ /___  / / \ \ 
 \___/____/_/_/  /_/_/ /_/ /_/  \___/    /_/\/   \_\ 
 
      github.com/SantanderMetGroup/climate4R



transformeR version 2.1.1 (2021-05-31) is loaded


Get the latest stable version (2.1.2) using <devtools::install_github('SantanderMetGroup/transformeR')>

Please see 'citation("transformeR")' to cite this package.

visualizeR version 1.6.0 (2020-05-23) is loaded


Get the latest stable version (1.6.1) using <devtools::install_github('SantanderMetGroup/visualizeR')>

Please see 'citation("visualizeR")' to cite this package.

geoprocessoR version 0.2.0 (2020-01-06) is loaded

Please see 'citation("geoprocessoR")' to cite this package.

rgdal: version: 1.5-12, (SVN revision 1018)
Geospatial Data Abstraction Library extensions to R successfully loaded
Loaded GDAL runtime: GDAL 2.2.2, released 2017/09/15
Path to GDAL shared files: /usr/share/gdal/2.2
GDAL binary built with GEOS: TRUE 
Loaded PROJ runtime: Rel. 4.8.0, 6 March 2012, [PJ_VERSION: 480]
Path to PROJ shared files: (autodetected)
Linking to sp version:1.4-2

Loading required package: usethis



Error in get(genname, envir = envir) : object 'testthat_print' not found


**Functions**: 
Here an additional function is used `climate4R.chunk` for loading and processing the data eficiently. To load the function in the working environment use the `source` R base function as follows.

In [2]:
# Load function for latitudinal chunking 
source_url("https://github.com/SantanderMetGroup/climate4R/blob/devel/R/climate4R.chunk.R?raw=TRUE")

SHA-1 hash of file is feed646847e95e5c030a0949248ddfd5001a1ae4



### 2. Parameter setting 

Set the parameters of the different functions used in this notebook.

In [1]:
# Select number of chunks
# Note: chunking sequentially splits the task into manageable data chunks to avoid memory problems
# Chunking operates by spliting the data into a predefined number latitudinal slices (n=2 in this example).
# Further details: https://github.com/SantanderMetGroup/climate4R/tree/master/R 
n.chunks <- 2
# Index, scenario, season and reference and future period(s) of interest, e.g.:
var <- "pr"  # index (maximum temperature)
project <- "CMIP5"
scenario <- "rcp85"  # scenario
season <- c(12, 1, 2)  # (entire year: season = 1:12; boreal winter (DJF): season = c(12, 1, 2); boreal summer (JJA): season = 6:8, and so on...)
base.period <- 1986:2005
future.period <- 2041:2060
# The target region 
lonLim <- c(-10, 35)
latLim <- c(35, 75)

### 3. Extract and select the ensemble models



In [4]:
dataset.hist <- UDG.datasets(paste0(project, ".*historical"))
dataset.ssp <- UDG.datasets(paste0(project, ".*", scenario))

Matches found for: CMIP5_AR5_1run CMIP5_AR5 CMIP5_subset

Label names are returned, set argument full.info = TRUE to get more information

Matches found for: CMIP5_AR5_1run CMIP5_AR5 CMIP5_subset

Label names are returned, set argument full.info = TRUE to get more information



We select the set of models CMIP5_AR5_1run

In [5]:
dataset.hist <- dataset.hist[["CMIP5_AR5_1run"]]
dataset.ssp <- dataset.ssp[["CMIP5_AR5_1run"]]

Check the available models:

In [6]:
dataset.ssp

[1] "CMIP5_ACCESS1-0_r1i1p1_rcp85"      "CMIP5_ACCESS1-3_r1i1p1_rcp85"     
 [3] "CMIP5_bcc-csm1-1-m_r1i1p1_rcp85"   "CMIP5_bcc-csm1-1_r1i1p1_rcp85"    
 [5] "CMIP5_BNU-ESM_r1i1p1_rcp85"        "CMIP5_CanESM2_r1i1p1_rcp85"       
 [7] "CMIP5_CCSM4_r1i1p1_rcp85"          "CMIP5_CESM1-BGC_r1i1p1_rcp85"     
 [9] "CMIP5_CMCC-CESM_r1i1p1_rcp85"      "CMIP5_CMCC-CM_r1i1p1_rcp85"       
[11] "CMIP5_CMCC-CMS_r1i1p1_rcp85"       "CMIP5_CNRM-CM5_r1i1p1_rcp85"      
[13] "CMIP5_CSIRO-Mk3-6-0_r1i1p1_rcp85"  "CMIP5_EC-EARTH_r1i1p1_rcp85"      
[15] "CMIP5_FGOALS-g2_r1i1p1_rcp85"      "CMIP5_GFDL-CM3_r1i1p1_rcp85"      
[17] "CMIP5_GFDL-ESM2G_r1i1p1_rcp85"     "CMIP5_GFDL-ESM2M_r1i1p1_rcp85"    
[19] "CMIP5_HadGEM2-CC_r1i1p1_rcp85"     "CMIP5_HadGEM2-ES_r1i1p1_rcp85"    
[21] "CMIP5_inmcm4_r1i1p1_rcp85"         "CMIP5_IPSL-CM5A-LR_r1i1p1_rcp85"  
[23] "CMIP5_IPSL-CM5A-MR_r1i1p1_rcp85"   "CMIP5_IPSL-CM5B-LR_r1i1p1_rcp85"  
[25] "CMIP5_MIROC-ESM-CHEM_r1i1p1_rcp85" "CMIP5_MIROC-ESM_r1i1p1_rcp85"     
[27] "CMIP5_MIROC5_r1i1p1_rcp85"         "CMIP5_MPI-ESM-LR_r1i1p1_rcp85"    
[29] "CMIP5_MPI-ESM-MR_r1i1p1_rcp85"     "CMIP5_MRI-CGCM3_r1i1p1_rcp85"     
[31] "CMIP5_NorESM1-M_r1i1p1_rcp85"

For brevity we will only use the first 6 models. 

To ensure that we get the same ensemble of models for both the historical and the future scenario we define the model names as follows:

In [7]:
models <- gsub(paste0("_", scenario), "", dataset.ssp[1:6])
models

[1] "CMIP5_ACCESS1-0_r1i1p1"    "CMIP5_ACCESS1-3_r1i1p1"   
[3] "CMIP5_bcc-csm1-1-m_r1i1p1" "CMIP5_bcc-csm1-1_r1i1p1"  
[5] "CMIP5_BNU-ESM_r1i1p1"      "CMIP5_CanESM2_r1i1p1"

Search the position of each model in `dataset.hist` and `dataset.ssp` using function `grep` in a loop. Subsequently subset the corresponding datasets.

In [8]:
# Search the position of each model in dataset.hist and dataset.ssp using function grep in a loop.
model.ind.hist <- vapply(models, grep, numeric(length = 1), x = dataset.hist)
model.ind.ssp <- vapply(models, grep, numeric(length = 1), x = dataset.ssp)
# Subset the datasets.
dataset.hist <- dataset.hist[model.ind.hist]
dataset.ssp <- dataset.ssp[model.ind.ssp]
dataset.hist; dataset.ssp

[1] "CMIP5_ACCESS1-0_r1i1p1_historical"   
[2] "CMIP5_ACCESS1-3_r1i1p1_historical"   
[3] "CMIP5_bcc-csm1-1-m_r1i1p1_historical"
[4] "CMIP5_bcc-csm1-1_r1i1p1_historical"  
[5] "CMIP5_BNU-ESM_r1i1p1_historical"     
[6] "CMIP5_CanESM2_r1i1p1_historical"

[1] "CMIP5_ACCESS1-0_r1i1p1_rcp85"    "CMIP5_ACCESS1-3_r1i1p1_rcp85"   
[3] "CMIP5_bcc-csm1-1-m_r1i1p1_rcp85" "CMIP5_bcc-csm1-1_r1i1p1_rcp85"  
[5] "CMIP5_BNU-ESM_r1i1p1_rcp85"      "CMIP5_CanESM2_r1i1p1_rcp85"

### 4. Load data

In [ ]:
hist <- lapply(dataset.hist, function(i) 
    climate4R.chunk(n.chunks = n.chunks,
                        C4R.FUN.args = list(FUN = "climatology",
                                            grid = list(dataset = i, var = var)),
                        loadGridData.args = list(years = base.period, season = season, lonLim = lonLim, latLim = latLim)))

In [ ]:
ssp <- lapply(dataset.ssp, function(i) 
    climate4R.chunk(n.chunks = n.chunks,
                        C4R.FUN.args = list(FUN = "climatology",
                                            grid = list(dataset = i, var = var)),
                        loadGridData.args = list(years = future.period, season = season, lonLim = lonLim, latLim = latLim)))

### 5. Build the ensemble 
Interpolate to the common grid and build the ensemble:

In [ ]:
# Inerpolate
common.grid <- loadGridData("../reference-grids/land_sea_mask_2degree.nc4", var = "sftlf", lonLim = lonLim, latLim = latLim)
hist.i <- lapply(hist, function(i) interpGrid(i, getGrid(common.grid), method = "bilinear"))
ssp.i <- lapply(ssp, function(i) interpGrid(i, getGrid(common.grid), method = "bilinear"))
# Ensemble building
hist.ens <- bindGrid(hist.i, dimension = "member")
ssp.ens <- bindGrid(ssp.i, dimension = "member")

### 6. Export to NetCDF

In [ ]:
gri2nc(hist.ens, "auxiliary-material/CMIP5_historical_pr.nc")
gri2nc(ssp.ens, "auxiliary-material/CMIP5_rcp85_pr.nc")

### 7. Session Information

In [ ]:
sessionInfo()